# Building AI Agents: A Beginner's Guide

Welcome! This notebook teaches you how to build a simple AI agent
using Python, Jupyter Notebook, and the OpenAI API.

## Learning objectives

By the end, you will understand:

- What an AI agent is
- How an LLM works with Python tools
- How function calling works
- How to implement an agent loop
- How to test and improve an agent

**Project:** Build a Math Assistant Agent that uses a Python
function to calculate averages.

### What is an AI agent?

An AI agent combines a language model with instructions and
potentially tools, memory, and a decision loop.

A typical workflow is:

1. Receive a user request.
2. Ask the model to decide what to do.
3. Execute a requested tool in Python.
4. Send the result back to the model.
5. Return a final answer.

## 1. Understand the architecture

The model does not automatically execute Python functions.
Instead, it requests a tool call. Your Python program validates
the request, executes the function, and returns the result.

User → AI model → Tool request → Python function
     → Tool result → AI model → Final answer

This separation is important for security and reliability.

## 2. Install and configure the environment

Run the following installation command in a notebook cell.

In [1]:
%pip install -U openai

Note: you may need to restart the kernel to use updated packages.


### Configure your API key

Create an API key in the OpenAI developer platform and set it
as the `OPENAI_API_KEY` environment variable before launching
Jupyter.

Do not paste your real API key into this notebook or commit it
to version control.

For macOS/Linux, run this in a terminal:

```bash
export OPENAI_API_KEY="your_api_key_here"
```

For Windows PowerShell:

```powershell
$env:OPENAI_API_KEY="your_api_key_here"
```

Restart Jupyter from that terminal if necessary.

In [2]:
import os

os.environ["OPENAI_API_KEY"] = "your_api_key_here"

if not os.getenv("OPENAI_API_KEY"):
    print("API key not found. Set OPENAI_API_KEY before running the agent.")
else:
    print("API key found.")

API key found.


## 3. Create your first tool

A tool is a normal Python function that performs a specific task.
Let's build a function that calculates an average.

In [3]:
def calculate_average(numbers: list[float]) -> float:
    """Calculate the average of a non-empty list of numbers."""
    if not numbers:
        raise ValueError("The list cannot be empty.")

    return sum(numbers) / len(numbers)


# Test the function
calculate_average([10, 20, 30])

20.0

Expected output:

```text
20.0
```

Notice that the function is deterministic: the same valid input
produces the same result. This makes it a useful tool for an agent.

## 4. Describe the tool to the AI model

The model needs a schema describing the tool name, purpose,
arguments, and expected argument types.

In [4]:
tools = [
    {
        "type": "function",
        "name": "calculate_average",
        "description": "Calculate the average of a list of numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "numbers": {
                    "type": "array",
                    "items": {"type": "number"},
                    "description": "Numbers to average"
                }
            },
            "required": ["numbers"],
            "additionalProperties": False
        },
        "strict": True
    }
]

tools

[{'type': 'function',
  'name': 'calculate_average',
  'description': 'Calculate the average of a list of numbers.',
  'parameters': {'type': 'object',
   'properties': {'numbers': {'type': 'array',
     'items': {'type': 'number'},
     'description': 'Numbers to average'}},
   'required': ['numbers'],
   'additionalProperties': False},
  'strict': True}]

The schema tells the model how to request the function. It does
not execute the function or grant the model direct Python access.

## 5. Connect to the OpenAI API

The Responses API accepts your request and tool definitions.
The model may return a normal answer or request a tool call.

Change `MODEL` if necessary to a model available to your account.
API access and usage charges may apply.

In [5]:
import json
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-5.4"

## 6. Build the agent loop

This is the core of the project.

The loop:
1. Sends the user request to the model.
2. Detects function calls.
3. Validates and executes each tool request.
4. Sends tool results back to the model.
5. Returns the final answer when no more tool calls are needed.

We also include a maximum iteration limit so the agent cannot
loop indefinitely.

In [6]:
def run_agent(question: str, max_steps: int = 5) -> str:
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string.")

    response = client.responses.create(
        model=MODEL,
        instructions=(
            "You are a helpful math assistant. "
            "Use calculate_average for average calculations. "
            "Explain results clearly. Do not invent tool results."
        ),
        input=question,
        tools=tools
    )

    for _ in range(max_steps):
        tool_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if not tool_calls:
            return response.output_text

        outputs = []

        for call in tool_calls:
            if call.name != "calculate_average":
                raise ValueError(f"Unknown tool: {call.name}")

            args = json.loads(call.arguments)
            numbers = args.get("numbers")

            if not isinstance(numbers, list) or not numbers:
                raise ValueError("Expected a non-empty list of numbers.")

            if any(
                isinstance(n, bool) or not isinstance(n, (int, float))
                for n in numbers
            ):
                raise ValueError("All values must be numbers.")

            if any(not __import__("math").isfinite(n) for n in numbers):
                raise ValueError("All values must be finite numbers.")

            result = calculate_average(numbers)

            outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps({"average": result})
            })

        response = client.responses.create(
            model=MODEL,
            instructions=(
                "You are a helpful math assistant. "
                "Explain tool results accurately and clearly."
            ),
            previous_response_id=response.id,
            input=outputs,
            tools=tools
        )

    raise RuntimeError("Agent reached its maximum number of steps.")

## 7. Run your first AI agent

Run this cell after configuring your API key.

The model should recognize that it needs the calculation tool,
request the calculation, and explain the returned result.

In [ ]:
answer = run_agent(
    "Calculate the average of 10, 20, 30, and 40."
)

print(answer)

Expected result: an answer explaining that the average is **25**.
The exact wording may differ.

## 8. Experiment with your agent

Try different requests. The model should use the tool when
a calculation is needed.

In [ ]:
print(run_agent("Find the average of 5, 15, 25, and 35."))

In [ ]:
print(run_agent("Explain what an average means in simple terms."))

### Exercise 1

Ask the agent to calculate the average of these values:

`12, 18, 24, 30`

Write your own request in the cell below.

In [ ]:
# TODO: Replace this question with your own
# print(run_agent("Calculate the average of ..."))

## 9. Challenge: Add a second tool

An agent becomes more useful when it can choose among multiple
well-defined tools.

Try extending the project with a `multiply_numbers` function.

Suggested steps:

1. Write and test the Python function.
2. Add its function schema to `tools`.
3. Update the agent loop to recognize the new tool.
4. Test multiplication and average requests.
5. Test an unexpected tool name to ensure it is rejected.

**Important:** A tool schema alone does not implement the tool.
Your Python code must explicitly validate and execute each
supported tool.

## 10. Best practices

### Security
- Keep API keys in environment variables or a secret manager.
- Validate all model-generated tool arguments.
- Give tools only the permissions they need.
- Require human approval for consequential actions.

### Reliability
- Limit the number of tool-call iterations.
- Handle API errors and rate limits in production.
- Test empty, invalid, and unexpected inputs.
- Do not assume the model's text is always correct.

### Cost and performance
- Minimize unnecessary model calls.
- Choose a model appropriate to the task.
- Log tool usage and measure response quality.

### Important distinction

This is a small educational example, not a production-ready
agent framework. A production system also needs robust error
handling, monitoring, tests, and an appropriate security design.

## 11. Summary

You have learned how to:

- Define a Python tool
- Describe a tool with a function schema
- Send a request to an LLM
- Execute model-requested tool calls
- Return tool results to the model
- Implement a bounded agent loop

### Final knowledge check

1. What is the difference between a chatbot and a tool-using agent?
2. Why does the model need a tool schema?
3. Who executes the Python function?
4. Why should tool arguments be validated?
5. Why should an agent loop have a maximum step count?

**Next step:** Explore the
[OpenAI Agents SDK for Python](https://openai.github.io/openai-agents-python/)
when you are ready to learn about structured orchestration,
handoffs, and tracing.